# Batch #4 - Off-diagonal emphasis screen (single-model, no code change)

**Where this comes from.** Recipe and architecture screens are flat (batch 2), the AUX head is flat,
threshold calibration is flat, and ensembles/soups top out around .768. The error analysis of the best
single model (A160_s42, .7476) says the remaining *systematic* errors are quadrant errors: the model
reads hate/offense, not stereotype.

Val errors of A160_s42 by quadrant (stereotype|hate_speech):

| quadrant | val rows | errors |
|---|---|---|
| yes\|no | 43 | **22 FN (51%)** |
| yes\|yes_implicit | 80 | **34 FN (42.5%)** |
| yes\|yes_explicit | 153 | 36 FN (23.5%) |
| no\|no | 300 | 45 FP |
| no\|yes_implicit | 95 | } 36 FP combined |
| no\|yes_explicit | 67 | } |

The training signal is 65% "hate-y" on positives and the model takes the shortcut. The fix that has
*not* been tried: **importance-reweight the off-diagonal quadrants** so the model has to learn the
boundary instead of the prior. The cheapest honest way to do that without touching pinned training
code is row duplication via `--train-extras` (the trainer appends extras AFTER the split, so val
videos are never touched; every extra row is a verbatim copy of a *train* row).

**Arms** (seed 42/43/44 each; identical A160 recipe otherwise):

| arm | extras (duplicated train rows, from 42-split train) | k |
|---|---|---|
| A160r_s* | *(in-session control re-run - drift check vs batch 1)* | 0 |
| E_r3_s* | `yes|no` x3 - the surface-positive stereotype rows | +896 |
| E_cs2_s* | `no|yes_*` x2 - hate-but-not-stereotype (counter-speech FPs) | +1420 |
| E_r3cs2_s* | both above | +2316 |
| E_nosl2_s* | (`yes|no` u `yes|implicit`) x2 - all no-explicit-slur positives | +1251 |

**How to read it.** Paired same-seed deltas vs A160 (cluster bootstrap over 56 val videos), plus the
quadrant-level FN/FP breakdown. A win needs to show up as: fewer FN on `yes|*` (or fewer FP on
`no|yes_*`) without new FP/FN elsewhere. Any single-shot +/-1pt on 738 rows is inside noise; the
screen bets on >=1.5-2pt pooled over 3 seeds. Caveats: duplication also adds optimizer steps per
epoch (inherent to the intervention); recorded absolutely-latest comparisons live in the batch-4
analysis script.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/Amn-O-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'  # same pin as batch 1-3: batch 4 is data-only


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data + GPU check

Attach `amnpham/lgbtqia` (Add Input), accelerator GPU T4 x2, Internet On. Builds the split-42
train/val frames and saves the val-side arrays the local analysis reads back
(labels, langs, videos, and the stereotype|hate quadrant per val row).


In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
tok = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738 and int(labs.sum()) == 276, \
    f'unexpected val frame: n={len(labs)} pos={int(labs.sum())}'
np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads.npy', quads)
print('val:', len(df_val), 'rows | pos', int(labs.sum()), '| clusters', df_val['yt_title'].nunique(),
      flush=True)
print('  val quadrants', dict(pd.Series(quads).value_counts()), flush=True)
qtr = df_train['stereotype'].astype(str) + '|' + df_train['hate_speech'].astype(str)
print('  train quadrants', dict(qtr.value_counts()), flush=True)


## 2. Extras files (the whole intervention)

Verbatim duplicates of TRAIN rows, selected by quadrant, written as per-language TSVs so the loader
re-derives the right prompt language. Names match `_(EN|IT|NL)_training.tsv` so `load_frame_from_file`
picks the language from the filename. `QUOTE_ALL` matches the corpus quoting (`data.py:_read_tsv`).


In [ ]:
import csv
import re

raw = {}
for p in data_mod.find_data_files(os.environ['LGBT_DATA_DIR']):
    m = re.search(r'_([A-Z]{2})_training\.tsv$', os.path.basename(p))
    if m and m.group(1) in ('EN', 'IT', 'NL'):
        raw[m.group(1)] = data_mod._read_tsv(p)
raw_all = pd.concat(raw.values(), ignore_index=True)

train_ids = set(df_train['StereoQueerEval_id'].astype(str))
val_ids = set(df_val['StereoQueerEval_id'].astype(str))
raw_tr = raw_all[raw_all['StereoQueerEval_id'].astype(str).isin(train_ids)].copy()
assert len(raw_tr) == len(df_train), (len(raw_tr), len(df_train))
assert not raw_tr['StereoQueerEval_id'].astype(str).isin(val_ids).any()
raw_tr['lang'] = raw_tr['StereoQueerEval_id'].astype(str).str.extract(r'training_([A-Z]{2})_')

quad = raw_tr['stereotype'].astype(str) + '|' + raw_tr['hate_speech'].astype(str)
GROUPS = {
    'r3': quad == 'yes|no',                                    # 448 rows, 6.5%  (24% of val FNs)
    'cs2': quad.isin(['no|yes_implicit', 'no|yes_explicit']),  # 1420 rows, 20.6% (counter-speech FPs)
    'nosl2': quad.isin(['yes|no', 'yes|yes_implicit']),        # 1251 rows, 18.2% (no explicit slur)
}
assert int(GROUPS['r3'].sum()) == 448 and int(GROUPS['cs2'].sum()) == 1420 \
    and int(GROUPS['nosl2'].sum()) == 1251, 'quadrant sizes changed - split or data moved'
K = {'r3': 2, 'cs2': 1, 'nosl2': 1}      # number of EXTRA copies (total multiplier = K + 1)
SPECS = {'r3': ('r3',), 'cs2': ('cs2',), 'r3cs2': ('r3', 'cs2'), 'nosl2': ('nosl2',)}

os.makedirs('extras', exist_ok=True)
EXTRAS = {}
for tag, groups in SPECS.items():
    chunks = [raw_tr[GROUPS[g]] for g in groups for _ in range(K[g])]
    extra = pd.concat(chunks, ignore_index=True)
    paths = []
    for lang in ('EN', 'IT', 'NL'):
        part = extra[extra['lang'] == lang].drop(columns='lang')
        p = f'extras/extra_{tag}_{lang}_training.tsv'
        part.to_csv(p, sep='\t', index=False, quoting=csv.QUOTE_ALL)
        paths.append(p)
    EXTRAS[tag] = ','.join(paths)
    print(f'{tag}: groups {groups} | +{len(extra)} extra rows '
          f'| train {len(df_train)} -> {len(df_train) + len(extra)}', flush=True)
print('EXTRAS:', EXTRAS, flush=True)


## 3. Run list and helpers

A160 recipe unchanged (`dropout 0.3, wd 0.01, head-lr 1e-3, xattn-lr 1e-4, backbone-lr 2e-5,
unfreeze 3, layers 3, shared encoder, no pooled-v, pool-init 0.02, 160/256, 15 epochs, patience 5`).
`A160r_s42/43/44` re-run the control *in this session* - if batch-4 controls reproduce batch-1 within
noise, the cross-session pairing used since batch 2 is validated; if they drift, we have a bug/drift
alarm before trusting any E-arm delta.


In [ ]:
BASE = dict(dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
            backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
            pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
            max_len=160, video_max_len=256, no_amp=False)

RUNS = []
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'A160r_s{s}', seed=s, split_seed=42))     # in-session control
for tag in ('r3', 'cs2', 'r3cs2', 'nosl2'):
    for s in (42, 43, 44):
        RUNS.append(dict(BASE, tag=f'E_{tag}_s{s}', seed=s, split_seed=42, extras=EXTRAS[tag]))

RECORDED = {'A160_s42': 0.7476, 'A160_s43': 0.7408, 'A160_s44': 0.7295}   # batch-1, same recipe+seed


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if os.path.exists(hist):
        print(f"-- skip (history exists): {hist}", flush=True)
        return True
    extras = f"--train-extras {r['extras']} " if r.get('extras') else ''
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"{'--no-amp ' if r['no_amp'] else ''}"
        f"{extras}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    best = float(h.loc[bi, 'val_macro_f1'])
    rec = RECORDED.get(r['tag'].replace('A160r_', 'A160_'))
    extra = f' | batch-1 same-seed {rec:.4f} (delta {best - rec:+.4f})' if rec else ''
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {best:.4f} | acc {h.loc[bi, 'val_acc']:.4f}{extra} | {len(h)} epochs",
          flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if not os.path.exists(hist) or not os.path.exists(out):
        print(f'-- no probs for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        model_xattn.DEFAULT_BACKBONE, unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(df_val, tok, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(f"weights/{r['tag']}_val_probs.npy", o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload-eval macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)


## 4. Run everything (~4-5 h on T4 x2)

15 runs: 3 in-session controls + 4 arms x 3 seeds. Resume-safe (history-exists skip); partial results
zipped after every run.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} (elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        zip_small()
    except Exception as e:                                    # noqa: BLE001
        print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
print(f'all {len(RUNS)} runs finished in {(time.time() - t0) / 3600:.2f} h', flush=True)


## 5. In-notebook summary

History table + per-quadrant FN/FP for every run + the 3-seed ensemble of each arm. The rigorous
paired statistics (cluster bootstrap vs controls, seed-pooled deltas) run locally on the pulled
arrays in `batch4_analysis.py`.


In [ ]:
rows = []
for r in RUNS:
    hist = f"weights/{r['tag']}_history.csv"
    if not os.path.exists(hist):
        continue
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    rows.append(dict(tag=r['tag'], seed=r['seed'], epochs=len(h),
                     best_epoch=int(h.loc[bi, 'epoch']),
                     best_f1=float(h.loc[bi, 'val_macro_f1']),
                     best_acc=float(h.loc[bi, 'val_acc'])))
summ = pd.DataFrame(rows).sort_values('tag')
summ.to_csv('weights/batch4_summary.csv', index=False)
print(summ.to_string(index=False), flush=True)

labs = np.load('weights/val_labels.npy')
quads = np.load('weights/val_quads.npy')


def quad_line(probs):
    pred = (probs >= 0.5).astype(int)
    out = []
    for q in ('yes|no', 'yes|yes_implicit', 'yes|yes_explicit',
              'no|no', 'no|yes_implicit', 'no|yes_explicit'):
        m = quads == q
        n = int(m.sum())
        if q.startswith('yes'):
            out.append(f'{q} FN {int((pred[m] == 0).sum()):>2}/{n:<3}')
        else:
            out.append(f'{q} FP {int((pred[m] == 1).sum()):>2}/{n:<3}')
    return ' | '.join(out)


probs_by_tag = {}
for r in RUNS:
    p = f"weights/{r['tag']}_val_probs.npy"
    if not os.path.exists(p):
        print(f'missing probs: {r["tag"]}', flush=True)
        continue
    probs_by_tag[r['tag']] = np.load(p)
    m = tx.metrics_from((probs_by_tag[r['tag']] >= 0.5).astype(int).tolist(), labs.tolist())
    print(f"{r['tag']:<14} macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f} | {quad_line(probs_by_tag[r['tag']])}",
          flush=True)

print()
for fam in ('A160r', 'E_r3', 'E_cs2', 'E_r3cs2', 'E_nosl2'):
    seeds = [t for t in probs_by_tag if t.startswith(fam + '_s')]
    if len(seeds) >= 2:
        ens = np.mean([probs_by_tag[t] for t in seeds], axis=0)
        m = tx.metrics_from((ens >= 0.5).astype(int).tolist(), labs.tolist())
        print(f"{fam:<8} x{len(seeds)} ensemble macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}",
              flush=True)

zip_small()
print('BATCH4 COMPLETE', flush=True)
